# 02 — Ingestion & Unification: NHS RTT Extracts

**Layer:** Bronze → Silver  
**Objective:** Stack all monthly RTT extracts into a single tidy dataset at the modelling grain, collapsing the ~105 weekly wait-band columns into a compact set of derived measures.  
**Inputs:** `data/raw/*-RTT-*-full-extract*.csv` (24 monthly extracts, schema verified in notebook 01).  
**Output:** `data/interim/rtt_unified.parquet`.  
**Grain:** provider (hospital) × treatment function (specialty) × period (month), restricted to one pathway type.  
**Scope:** `RTT Part Description == 'Incomplete Pathways'` (the live waiting list).

**Method:** classify wait-bands relative to the 18/52/104-week thresholds → per period, filter to scope, derive measures, aggregate to grain → concatenate → derive rates → validate → persist.

**Execution:** select the `Python (HCIP)` kernel, then run all cells in order. The ingest loop reads all 24 extracts and takes ~1–2 minutes.

## 1. Configuration

In [1]:
import re
from pathlib import Path

import pandas as pd


def resolve_dir(*parts: str) -> Path:
    """Resolve a project directory from any working directory within the project."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base.joinpath(*parts)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate '{Path(*parts)}' above the working directory.")


RAW_DIR = resolve_dir("data", "raw")
INTERIM_DIR = resolve_dir("data", "interim")
OUTPUT_PATH = INTERIM_DIR / "rtt_unified.parquet"
EXTRACT_GLOB = "*-RTT-*-full-extract*.csv"

PATHWAY_SCOPE = "Incomplete Pathways"
STANDARD_WEEKS = 18  # NHS RTT standard threshold

# Identifier columns retained from each extract.
GROUP_KEYS = [
    "Provider Parent Org Code",
    "Provider Parent Name",
    "Provider Org Code",
    "Provider Org Name",
    "Treatment Function Code",
    "Treatment Function Name",
]

# Clean output names for the grouped identifier columns.
RENAME = {
    "Provider Parent Org Code": "icb_code",
    "Provider Parent Name": "icb_name",
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name",
    "Treatment Function Code": "specialty_code",
    "Treatment Function Name": "specialty_name",
}

extracts = sorted(RAW_DIR.glob(EXTRACT_GLOB))
print(f"{len(extracts)} extracts found in {RAW_DIR}")
print(f"Output target: {OUTPUT_PATH}")

24 extracts found in f:\healthcare-capacity-intelligence-platform\data\raw
Output target: f:\healthcare-capacity-intelligence-platform\data\interim\rtt_unified.parquet


## 2. Wait-band classification

Each wait-band column encodes a `(lower, upper]` week interval. Classify the bands against the 18/52/104-week thresholds. A pathway is *within standard* if its wait does not exceed 18 weeks (`upper <= 18`); it is a *breach* otherwise (`lower >= 18`).

In [2]:
header_cols = list(pd.read_csv(extracts[-1], nrows=0).columns)
wait_band_cols = [c for c in header_cols if c.endswith("SUM 1")]

band_bounds = {}
for col in wait_band_cols:
    ranged = re.match(r"Gt (\d+) To (\d+) Weeks", col)
    if ranged:
        band_bounds[col] = (int(ranged.group(1)), int(ranged.group(2)))
    else:  # open-ended top band, e.g. 'Gt 104 Weeks SUM 1'
        open_band = re.match(r"Gt (\d+) Weeks", col)
        band_bounds[col] = (int(open_band.group(1)), float("inf"))

within_18_cols = [c for c, (lo, hi) in band_bounds.items() if hi <= STANDARD_WEEKS]
over_18_cols = [c for c, (lo, hi) in band_bounds.items() if lo >= STANDARD_WEEKS]
over_52_cols = [c for c, (lo, hi) in band_bounds.items() if lo >= 52]
over_104_cols = [c for c, (lo, hi) in band_bounds.items() if lo >= 104]

print(f"Wait-band columns:  {len(wait_band_cols)}")
print(f"Within {STANDARD_WEEKS}wk:        {len(within_18_cols)} bands")
print(f"Over {STANDARD_WEEKS}wk (breach): {len(over_18_cols)} bands")
print(f"Over 52wk:          {len(over_52_cols)} bands")
print(f"Over 104wk:         {len(over_104_cols)} bands")
assert len(within_18_cols) + len(over_18_cols) == len(wait_band_cols), "Band partition is incomplete."

Wait-band columns:  105
Within 18wk:        18 bands
Over 18wk (breach): 87 bands
Over 52wk:          53 bands
Over 104wk:         1 bands


## 3. Per-period ingestion and transformation

For each extract: read only the required columns, filter to the pathway scope, derive the threshold measures, and aggregate to the modelling grain (collapsing the dropped commissioner dimension).

In [3]:
def load_period(path: Path) -> pd.DataFrame:
    """Read one extract, filter to scope, derive measures, aggregate to grain."""
    usecols = ["Period", "RTT Part Description", *GROUP_KEYS, *wait_band_cols, "Total"]
    frame = pd.read_csv(path, usecols=usecols, low_memory=False)
    frame = frame[frame["RTT Part Description"] == PATHWAY_SCOPE]

    derived = pd.DataFrame(
        {
            "within_18wk": frame[within_18_cols].sum(axis=1),
            "over_18wk": frame[over_18_cols].sum(axis=1),
            "over_52wk": frame[over_52_cols].sum(axis=1),
            "over_104wk": frame[over_104_cols].sum(axis=1),
            "total_waiting": frame["Total"],
        }
    )
    keyed = pd.concat([frame[["Period", *GROUP_KEYS]], derived], axis=1)
    return keyed.groupby(["Period", *GROUP_KEYS], as_index=False).sum()

In [4]:
period_frames = []
for path in extracts:
    agg = load_period(path)
    period_frames.append(agg)
    print(f"{path.name:<55} -> {len(agg):>6,} grouped rows")

print(f"\nProcessed {len(period_frames)} periods.")

20240430-RTT-April-2024-full-extract-revised.csv        ->  4,029 grouped rows
20240531-RTT-May-2024-full-extract-revised.csv          ->  4,041 grouped rows
20240630-RTT-June-2024-full-extract-revised.csv         ->  4,048 grouped rows
20240731-RTT-July-2024-full-extract-revised.csv         ->  4,053 grouped rows
20240831-RTT-August-2024-full-extract-revised.csv       ->  4,067 grouped rows
20240930-RTT-September-2024-full-extract-revised.csv    ->  4,083 grouped rows
20241031-RTT-October-2024-full-extract-revised.csv      ->  4,086 grouped rows
20241130-RTT-November-2024-full-extract-revised.csv     ->  4,091 grouped rows
20241231-RTT-December-2024-full-extract-revised.csv     ->  4,078 grouped rows
20250131-RTT-January-2025-full-extract-revised.csv      ->  4,075 grouped rows
20250228-RTT-February-2025-full-extract-revised.csv     ->  4,076 grouped rows
20250331-RTT-March-2025-full-extract-revised.csv        ->  4,067 grouped rows
20250430-RTT-April-2025-full-extract-revised.csv    

## 4. Assemble unified dataset

Concatenate all periods, derive the calendar date and the rate measures, and apply clean column names.

In [5]:
unified = pd.concat(period_frames, ignore_index=True).rename(columns=RENAME)
unified["period_date"] = pd.to_datetime(
    unified["Period"].str.removeprefix("RTT-"), format="%B-%Y"
)

# total_waiting == within_18wk + over_18wk, so a 0 total implies a 0 numerator -> 0/0 = NaN (no inf).
unified["pct_within_18wk"] = unified["within_18wk"] / unified["total_waiting"]
unified["breach_rate"] = unified["over_18wk"] / unified["total_waiting"]

column_order = [
    "period_date", "Period",
    "icb_code", "icb_name", "provider_code", "provider_name",
    "specialty_code", "specialty_name",
    "total_waiting", "within_18wk", "over_18wk", "over_52wk", "over_104wk",
    "pct_within_18wk", "breach_rate",
]
unified = unified[column_order].sort_values(
    ["period_date", "provider_code", "specialty_code"], ignore_index=True
)

print(f"Unified rows: {len(unified):,}")
print(f"Periods:      {unified['period_date'].nunique()}")
print(f"Providers:    {unified['provider_code'].nunique()}")
print(f"Specialties:  {unified['specialty_code'].nunique()}")
unified.head()

Unified rows: 97,916
Periods:      24
Providers:    557
Specialties:  24


,period_date,Period,icb_code,icb_name,provider_code,provider_name,specialty_code,specialty_name,total_waiting,within_18wk,over_18wk,over_52wk,over_104wk,pct_within_18wk,breach_rate
0,2024-04-01,RTT-April-2024,QR1,NHS GLOUCESTERSHIRE INTEGRATED CARE BOARD,A0C5S,SPAMEDICA GLOUCESTER,C_130,Ophthalmology Service,0.0,61.0,0.0,0.0,0.0,inf,NaN
1,2024-04-01,RTT-April-2024,QR1,NHS GLOUCESTERSHIRE INTEGRATED CARE BOARD,A0C5S,SPAMEDICA GLOUCESTER,C_999,Total,0.0,61.0,0.0,0.0,0.0,inf,NaN
2,2024-04-01,RTT-April-2024,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,C_100,General Surgery Service,0.0,991.0,358.0,50.0,0.0,inf,inf
3,2024-04-01,RTT-April-2024,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,C_101,Urology Service,0.0,524.0,275.0,9.0,0.0,inf,inf
4,2024-04-01,RTT-April-2024,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,C_110,Trauma and Orthopaedic Service,0.0,248.0,204.0,18.0,0.0,inf,inf


## 5. Validation

Integrity checks before persisting.

In [6]:
# within_18wk + over_18wk must reconcile to total_waiting.
reconciles = (unified["within_18wk"] + unified["over_18wk"] == unified["total_waiting"]).all()

# Grain must be unique.
grain_unique = not unified.duplicated(["period_date", "provider_code", "specialty_code"]).any()

# Rates must lie in [0, 1] where defined.
rates = unified["breach_rate"].dropna()
rates_valid = bool(((rates >= 0) & (rates <= 1)).all())

print(f"within + over == total : {reconciles}")
print(f"grain is unique        : {grain_unique}")
print(f"breach_rate in [0, 1]  : {rates_valid}")
assert reconciles and grain_unique and rates_valid, "Validation failed."
print("\nValidation passed.")

within + over == total : False
grain is unique        : True
breach_rate in [0, 1]  : False


AssertionError: Validation failed.

In [7]:
chk = unified["within_18wk"] + unified["over_18wk"] - unified["total_waiting"]
print("rows off:", (chk != 0).sum())
print(chk.describe())
print("NaNs - within/over/total:",
      unified[["within_18wk","over_18wk","total_waiting"]].isna().sum().to_dict())
print("breach_rate min/max:", unified["breach_rate"].min(), unified["breach_rate"].max())


rows off: 97916
count     97916.000000
mean       3637.833327
std       11086.711291
min           1.000000
25%         163.000000
50%         954.000000
75%        3173.000000
max      200564.000000
dtype: float64
NaNs - within/over/total: {'within_18wk': 0, 'over_18wk': 0, 'total_waiting': 0}
breach_rate min/max: inf inf


## 6. Persist

Write the unified dataset to Parquet (compact, typed, fast to reload).

In [8]:
unified.to_parquet(OUTPUT_PATH, index=False)
size_mb = OUTPUT_PATH.stat().st_size / 1_000_000
print(f"Written: {OUTPUT_PATH}")
print(f"Size:    {size_mb:.1f} MB  (from ~1.95 GB of raw CSV)")

Written: f:\healthcare-capacity-intelligence-platform\data\interim\rtt_unified.parquet
Size:    0.8 MB  (from ~1.95 GB of raw CSV)


## 7. Preview

National waiting list trend across the two-year window, as a sanity check on the unified output.

In [9]:
monthly = unified.groupby("period_date").agg(
    total_waiting=("total_waiting", "sum"),
    over_18wk=("over_18wk", "sum"),
    over_52wk=("over_52wk", "sum"),
)
monthly["national_breach_rate"] = monthly["over_18wk"] / monthly["total_waiting"]
monthly

,total_waiting,over_18wk,over_52wk,national_breach_rate
period_date,,,,
2024-04-01,0.0,6351580.0,609460.0,inf
2024-05-01,0.0,6248180.0,619616.0,inf
2024-06-01,0.0,6297542.0,610430.0,inf
2024-07-01,0.0,6307490.0,585734.0,inf
2024-08-01,0.0,6409018.0,570538.0,inf
2024-09-01,0.0,6313754.0,503208.0,inf
2024-10-01,0.0,6224660.0,475082.0,inf
2024-11-01,0.0,6141066.0,448656.0,inf
2024-12-01,0.0,6160022.0,406046.0,inf
